STEP 1- Import Useful Requirements

In [1]:
import os # use to access the operating system
from dotenv import load_dotenv # use for access API's Key from .env files

# Langchain
# for data load
from langchain_community.document_loaders import TextLoader
# for data split
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import JinaEmbeddings


C:\Users\JITU PRADHAN\AppData\Local\Temp\ipykernel_37268\3300576414.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("All key are loaded")

All key are loaded


Loading our Data

In [4]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

DATA INGESTION

In [5]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()
print("Data Loaded")
print("="*40)
print(documents)

Data Loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

In [6]:
len(documents)

1

In [7]:
print(f"Total number of carectors in documents: {len(documents[0].page_content)}")

Total number of carectors in documents: 2598


Split the data

In [8]:
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_spliter.split_documents(documents)

In [9]:
len(chunks)

9

Embadding Data Chunks

In [10]:
embedding_model = JinaEmbeddings(
    model_name = "jina-embeddings-v5-omni-small"
)
print(f"Embedding Model is ready {embedding_model.model_name}")

Embedding Model is ready jina-embeddings-v5-omni-small


In [11]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)

print("Chunk are stored", vector_store.index.ntotal)

Chunk are stored 9


In [12]:
test_query = "How many sick leaves employees get"

In [13]:
top_matchs = vector_store.similarity_search(test_query,k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matchs,start=1):
    print(f"----Match {i}-----")
    print(match.page_content)

Query: How many sick leaves employees get

----Match 1-----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
----Match 2-----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


Tools

In [19]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reibursement, code of conduct, holidays, or exit process
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

Data Retrival Process

In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

In [20]:
from langchain.agents import create_agent

hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up
    fatch before answering.
    If the answer isn't in the search results, say you don't know
    instead of guessing.
    """
)
print("LLM ready to answer")

LLM ready to answer


In [ ]:
def ask_hr_assistant(question: str) -> str:
    """
    Send a question to the RAG agent and print a nicely formatted answer.
    """
    print("="*60)
    print("QUESTION:",question)
    print("="*60)
    
    response = hr_assistant.invoke({
        "messages":[
            {
                "role":"user",
                "content":question
            }
        ]
    })
    
    answer= ("ANSWER:",answer)
    print("="*60)
    print()
    return answer
    

In [21]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"Tell me about leave policies how to apply from leave"
            }
        ]
    }
)

In [22]:
response

{'messages': [HumanMessage(content='Tell me about leave policies how to apply from leave', additional_kwargs={}, response_metadata={}, id='7fdb57bb-9a25-4e9f-9df5-b700f91b9f67'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use search_hr_policy tool to look up info about leave policies. The user asks: "Tell me about leave policies how to apply from leave". So we need to search.', 'tool_calls': [{'id': 'fc_b882a56a-00fb-4207-a960-3cad847fa9dc', 'function': {'arguments': '{"question":"leave policies how to apply for leave"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 208, 'total_tokens': 281, 'completion_time': 0.160454741, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.010230694, 'prompt_tokens_details': None, 'queue_time': 0.317101019, 'total_time': 0.170685435}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'servic

In [24]:
print(response["messages"][-1].content)

**Leave Policies – Quick Overview**

| Type of Leave | Entitlement | Key Rules |
|---------------|------------|-----------|
| **Annual / Paid Vacation** | 20 days per calendar year (full‑time) | • Submit a request through the HR portal **at least 5 working days** before the start date.<br>• Up to **5 unused days** can be carried forward to the next year; any excess lapses. |
| **Sick Leave** | 10 paid days per year | • No portal request needed for the first 2 consecutive days (just inform your manager).<br>• For sick leave **longer than 2 consecutive days**, attach a **medical certificate** when you log the request in the HR portal. |
| **Public Holidays** | 12 days (company‑wide) | • If you work on a public holiday you’re eligible for compensatory leave or overtime pay, per the holiday calendar released each year. |

---

### How to Apply for Leave

1. **Log into the HR Portal**  
   - Use your employee credentials to access the “Leave Management” section.

2. **Select the Leave Type*